> **Recorded research notebook.** Executed outputs are retained from the completed study. Empirical recomputation requires the separately retained input snapshots described in `data/README.md`. Run `python scripts/reproduce_public.py` for checks and a synthetic demonstration using only this public repository. Historical publication-stage notes below describe the notebook execution date; the final PDF is now in `paper/final/Information-Diffusion-Final-Paper.pdf`.

# 00 · Research question and design

**Question:** how does the association between a scheduled information release and quoted prices evolve over the following minutes? The project separates an observable response curve, a conditional adjustment model and a predictive lead–lag diagnostic. None alone observes the instant when every investor has absorbed every part of a release.

This is an executed research record before paper assembly. The empirical study uses free public data, 2018–2025 FOMC statements as its primary setting, minutes as an extension, and SPX, NSX and EURUSD provider bid quotes. Earnings calls remain a future extension: a call contains multiple release times and needs reliable free historical timestamps, expectations and corporate-action-aware prices.

As Fama (1991) explains in *Efficient capital markets: II*, empirical efficiency tests depend on the return model used to evaluate prices. As MacKinlay (1997) explains in *Event studies in economics and finance*, a credible event study begins with a precise event and window definition. These points motivate narrow windows, explicit information sets and restrained interpretation.

In [1]:
from pathlib import Path
import sys, json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'diffusion/common.py').exists())
if str(root) not in sys.path: sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
from diffusion.common import ROOT, design, sha256
from diffusion.reporting import table
from diffusion.presentation import show_table, show_json, show_figure, show_plot, equation
pd.set_option('display.max_columns', 12)
print('Project:', ROOT.name)
print('Clock: A02 audited seasonal mapping. Later sample: post-audit chronological transfer.')

Project: Information-Diffusion-Research
Clock: A02 audited seasonal mapping. Later sample: post-audit chronological transfer.


## Design that survives the audit

The original design is immutable. Amendment A01 changed the data route after slow daily-file acquisition; A02 repaired a documented clock mismatch. The original held-out run had already been viewed. Corrected 2024–2025 results are therefore **post-audit chronological transfer**, never an untouched holdout. The internal filename `holdout` is retained for traceability.

The primary quote series remains SPX even though other series have better historical coverage. Missing windows are excluded as whole event–asset pairs; no missing return is imputed. The 2023 sample is retained in the audit but is too small for the declared response regressions.

In [2]:
d = design()
keys = ['primary_asset','primary_event_type','assets','surprise_inputs','sample_gates']
show_json({k:d[k] for k in keys})
events = pd.read_csv(ROOT/'data/processed/event-register.csv')
counts = events.groupby(['phase','event_type']).size().reset_index(name='planned_events')
show_table(counts, 'Table N00.1. Scheduled releases before quote-coverage exclusions.')
assert len(events) == 128
assert ((events.phase=='development') & (events.event_type=='Statements')).sum() == 39
print('Event register verified: 63 statements and 65 minutes releases.')

{
  "primary_asset": "SPX",
  "primary_event_type": "Statements",
  "assets": {
    "SPX": "SPXUSD",
    "NSX": "NSXUSD",
    "EURUSD": "EURUSD"
  },
  "surprise_inputs": [
    "MP1",
    "MP2",
    "ED2",
    "ED3",
    "ED4"
  ],
  "sample_gates": {
    "minimum_development_events": 25,
    "minimum_validation_events": 6,
    "minimum_holdout_events": 10,
    "minimum_coverage_fraction": 0.8
  }
}

phase,event_type,planned_events
development,Minutes,40
development,Statements,39
holdout,Minutes,17
holdout,Statements,16
validation,Minutes,8
validation,Statements,8


Event register verified: 63 statements and 65 minutes releases.


## Three clocks and three questions

Calendar time identifies the public release; event time counts completed minute bars; the information clock records when a regressor becomes observable. The USMPD rate surprise is measured from −10 to +20 minutes. It cannot be used as a release-time predictor. Within-window fits are retrospective associations. The separate +20-to-+25 exercise grants idealised immediate availability at +20, without a publication-latency model.

As Gürkaynak, Sack and Swanson (2005) demonstrate, policy actions and statements can contain distinct target and path information. A single principal component is a compact descriptor, not their two-factor structural decomposition. As Jarociński and Karadi (2020) explain, announcements may also reveal information about economic conditions; a rate-price covariance does not isolate a pure policy shock.

In [3]:
timeline = pd.DataFrame([
    ['−61 to −1','Required pre-event bars','Quality checks and pre-event controls'],
    ['0','Scheduled statement or minutes release','Event-time origin'],
    ['1 to 20','Completed minute horizons','Primary retrospective response curve'],
    ['−10 to +20','USMPD surprise measurement','Full factor unavailable at release'],
    ['20 to 25','Post-observation exercise','Idealised +20 availability only'],
    ['+30, where applicable','Press conference begins','New information; extended plots are descriptive']
], columns=['time / minutes','observation','interpretation'])
show_table(timeline, 'Table N00.2. Release time, price observation and factor availability.')
equation(2)
equation(5)
equation(7)

time / minutes,observation,interpretation
−61 to −1,Required pre-event bars,Quality checks and pre-event controls
0,Scheduled statement or minutes release,Event-time origin
1 to 20,Completed minute horizons,Primary retrospective response curve
−10 to +20,USMPD surprise measurement,Full factor unavailable at release
20 to 25,Post-observation exercise,Idealised +20 availability only
"+30, where applicable",Press conference begins,New information; extended plots are descriptive


Equation (2). Cumulative event-time response.
Y_{e,a}(h)=10^4\log\!\left(P_{e,a,h-1}/P_{e,a,-1}\right)
A bar is labelled by its start minute. The close of bar h−1 is the horizon-h endpoint.

Equation (5). Unrestricted horizon regression.
Y_{e,a}(h)=\alpha_{a,h}+\beta_{a,h}z_e+\varepsilon_{e,a,h}
Separate OLS regressions estimate a response association at each of 20 horizons.

Equation (7). Predeclared lagged-response estimand.
L_{a,5:20}=\beta_{a,20}-\beta_{a,5}
Estimate directly by regressing Y(20)−Y(5) on the same event-level surprise.

## What would count as a speed estimate?

A visible return and a statistically nonzero response are not enough. A conditional speed is reported only if the terminal response is distinguishable from zero, the curve plausibly reaches a plateau, the monotone fit is adequate and interior, bootstrap fits are stable, and sample coverage passes. The plateau margin is estimated from the data; it is a reporting heuristic rather than a calibrated equivalence test. Failure to reject a late response is not evidence of complete adjustment.

As Hong, Lim and Stein (2000) motivate in their study of slow news diffusion, information can propagate unevenly across investors. Their longer-horizon equity evidence motivates a question here; it does not establish a minute-scale FOMC mechanism.

In [4]:
equation(8)
equation(9)
equation(10)
equation(18)
refs = json.loads((ROOT/'references/references.json').read_text())
print('Verified bibliography entries:', len(refs))
print('Full Harvard bibliography: references/harvard-references.txt')
print('Source relevance and verification notes: references/source-register.csv')

Equation (8). Conditional adjustment curve.
\beta(h)=A\left[1-(1-f)\exp\{-k(h-1)\}\right],\qquad h\geq1
A is a signed long-run model amplitude; f is the first-minute fraction and k is a rate per minute.

Equation (9). Time to a target adjustment fraction.
t_q=\begin{cases}1,&f\geq q,\\1+k^{-1}\log\!\left((1-f)/(1-q)\right),&f<q.\end{cases}
This is conditional on the monotone model; it is not a model-free time at which all information is incorporated.

Equation (10). Half-life of the remaining adjustment.
\tau_{1/2}=\frac{\log 2}{k}
The remaining-gap half-life differs from time to 50 percent of total adjustment when f is nonzero.

Equation (18). Plateau reporting diagnostic.
\mathrm{CI}_{0.90}(\beta_{20}-\beta_{10})\subset\left(-0.25|\hat\beta_{20}|,\ 0.25|\hat\beta_{20}|\right)
The margin depends on an estimated terminal response; this is a reporting heuristic, not a validated formal equivalence test.

Verified bibliography entries: 18
Full Harvard bibliography: references/harvard-references.txt
Source relevance and verification notes: references/source-register.csv


## Milestone outcome

The design, estimands, information sets and reporting rules are explicit. The exercise can measure conditional price-response paths and test for lagged associations. It cannot establish universal market efficiency, a causal structural shock, executable profit, or the exact sub-minute time when all information is priced.